# TD 2 : Données manquantes et anomalies (CORRECTION)

**BUT INFO3 Développeur IA, R5.C.08, IA5, Partie 1**
M. Sakkari, IUT de La Rochelle

> **Version enseignant :** codes complétés et réponses aux questions.

## Dans ce TD, on apprend à...
- comprendre **pourquoi** une case est vide dans un tableau de données, et ce qu'on peut faire ;
- repérer des valeurs **très bizarres** (par exemple un prix de 1 € pour une maison) ;
- décider **quoi en faire** : les supprimer, les corriger ou les garder.

## Les données
- **DVF** (*Demandes de Valeurs Foncières*) : la liste des ventes de logements à La Rochelle en 2024, publiée par l'État. Vous l'avez déjà vue au TD 1.
- **Avis Amazon** : des avis de clients sur des produits alimentaires (un seul exercice).

> **Comment travailler :** une cellule « Démo » est déjà écrite, il faut la lancer et la lire. Une cellule « À vous » est à compléter : remplacez les `...` par votre code.


## Petit lexique avant de commencer

| Mot | Ce que ça veut dire | Exemple |
|---|---|---|
| **Tableau** (*DataFrame*) | des données rangées en lignes et en colonnes, comme dans Excel | notre tableau DVF |
| **Ligne** | un élément décrit dans le tableau | une vente |
| **Colonne** (ou **variable**) | une information, la même pour toutes les lignes | le prix, la surface |
| **Valeur manquante** (`NaN`) | une case vide | une surface qui n'a pas été notée |
| **Moyenne** | on additionne les valeurs, puis on divise par leur nombre | moyenne de 10, 12 et 20 : (10 + 12 + 20) / 3 = 14 |
| **Médiane** | la valeur du milieu quand on trie les valeurs | médiane de 10, 12 et 20 : 12 |
| **Pourcentage** | une part sur 100 | 25 cases vides sur 100 = 25 % |

**Moyenne ou médiane ?** Prenons 5 surfaces de terrain : 100, 120, 130, 140 et 2 000 m². La moyenne vaut 498 m² : elle est tirée vers le haut par la valeur 2 000. La médiane vaut 130 m² : elle ne bouge presque pas. La médiane résiste mieux aux valeurs extrêmes.

## Les colonnes de DVF utilisées dans ce TD

| Colonne | Ce qu'elle contient |
|---|---|
| `id_mutation` | le numéro d'une vente. Une vente peut avoir **plusieurs lignes** : une par lot (un appartement, une cave, un parking...) |
| `valeur_fonciere` | le prix de la vente, en euros |
| `type_local` | le type de bien : Maison, Appartement, Dépendance (cave, garage...), Local industriel ou commercial |
| `surface_reelle_bati` | la surface construite, en m² |
| `surface_terrain` | la surface du terrain, en m² |


In [1]:
# Objectif : charger les données DVF (les ventes immobilières à La Rochelle en 2024) pour pouvoir les étudier.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dossier contenant les fichiers de données
DATA = "../data"

# Chargement du fichier DVF (transactions immobilières, La Rochelle 2024)
dvf = pd.read_csv(f"{DATA}/dvf_la_rochelle_2024.csv")
print("Dimensions :", dvf.shape)   # (nombre de lignes, nombre de colonnes)

Dimensions : (3854, 14)


# 1. Comprendre pourquoi une valeur manque

### Concept

Une case vide s'appelle une **valeur manquante** (`NaN`). Avant de la remplir, ou de supprimer la ligne, on se pose une question : **pourquoi cette case est-elle vide ?** La réponse change ce qu'on doit faire. Il y a quatre cas :

| Cas | Ce que ça veut dire | Exemple |
|---|---|---|
| **Hasard pur** (*MCAR*, Missing Completely At Random) | la case est vide par accident, sans lien avec les autres données | un capteur plante au hasard |
| **Lié à une autre colonne** (*MAR*, Missing At Random) | la case est vide plus souvent pour certaines lignes, et on le voit dans une autre colonne | la surface de terrain est vide surtout pour les appartements : on le voit dans la colonne `type_local` |
| **Lié à la valeur elle-même** (*MNAR*, Missing Not At Random) | la case est vide **à cause** de la valeur qu'elle aurait contenue | des personnes à très hauts revenus ne déclarent pas leur salaire |
| **Structurel** | la valeur **n'existe pas** | un appartement n'a pas de jardin à lui |

Pourquoi est-ce important ? Si on remplit toutes les cases vides de la même façon sans savoir pourquoi elles sont vides, on peut **déformer** les résultats.

Référence : D. B. Rubin, [*Inference and Missing Data*](https://dash.harvard.edu/handle/1/3408223), Biometrika, 1976.

### Démo : quelles colonnes ont des cases vides ?

On calcule, pour chaque colonne, le pourcentage de cases vides.

In [2]:
# Objectif : savoir quelles colonnes ont des cases vides, et en quelle proportion.

# Étape 1 : isna() répond True si la case est vide, False sinon
# Étape 2 : mean() fait la moyenne de ces True/False. True compte pour 1 et False pour 0,
#           donc la moyenne donne la proportion de cases vides
# Étape 3 : on multiplie par 100 pour avoir un pourcentage, on arrondit (round),
#           puis on trie (sort_values) de la colonne la plus vide à la moins vide
(dvf.isna().mean() * 100).round(1).sort_values(ascending=False)

surface_terrain              62.2
surface_reelle_bati          58.1
type_local                   24.3
nombre_pieces_principales    24.3
adresse_numero               14.2
adresse_nom_voie              0.5
code_postal                   0.5
longitude                     0.4
latitude                      0.4
date_mutation                 0.0
nature_mutation               0.0
valeur_fonciere               0.0
id_mutation                   0.0
nom_commune                   0.0
dtype: float64

**Lecture :** cinq colonnes ont beaucoup de cases vides (plus de 10 %) : `surface_terrain` (62 %), `surface_reelle_bati` (58 %), `type_local` et `nombre_pieces_principales` (24 %), `adresse_numero` (14 %).

Mais ce pourcentage ne dit pas **pourquoi** les cases sont vides. Pour le savoir, on regarde **où** elles sont vides : est-ce pareil pour une maison et pour un appartement ?

### À vous : le manque est-il le même pour tous les biens ?

On veut savoir si `surface_terrain` est vide aussi souvent pour une maison que pour un appartement.

Pour cela, on utilise `groupby`. Cette fonction **range les lignes en paquets** : un paquet pour les maisons, un paquet pour les appartements, etc. On peut ensuite faire un calcul sur chaque paquet.

**À faire :** calculez, pour chaque type de bien (`type_local`), le pourcentage de lignes où `surface_terrain` est vide.

**Questions :**
1. Quel type de bien a le plus de cases vides ? Lequel en a le moins ?
2. Pourquoi, selon vous ?

In [3]:
# Objectif : savoir si surface_terrain est vide aussi souvent pour tous les types de bien.
# Résultat attendu : un pourcentage de cases vides pour chaque type_local (Maison, Appartement...).

# Étape 1 : créer une colonne "terrain_manquant" qui vaut True si surface_terrain est vide
dvf["terrain_manquant"] = dvf["surface_terrain"].isna()

# Étape 2 : ranger les lignes en paquets par type de bien (groupby), puis calculer la moyenne
#           de "terrain_manquant" dans chaque paquet (mean) et multiplier par 100
taux_terrain = (dvf.groupby("type_local")["terrain_manquant"].mean() * 100).round(1)
taux_terrain

type_local
Appartement                                 87.8
Dépendance                                  81.6
Local industriel. commercial ou assimilé    60.6
Maison                                       3.5
Name: terrain_manquant, dtype: float64

### Réponse

```text
Appartement                      87,8 %
Dépendance                       81,6 %
Local industriel ou commercial   60,6 %
Maison                            3,5 %
```

**Question 1 : quel type de bien a le plus de manquants ? Et le moins ?**
Les appartements (88 %) en ont le plus. Les maisons (3,5 %) en ont le moins.

**Question 2 : pourquoi ?**
Un appartement n'a presque jamais de terrain à lui, donc la valeur n'existe pas et la case reste vide. C'est un manque **structurel**. Pour une maison, un terrain vide est presque toujours un oubli.

Le manque dépend du type de bien, une information qu'on connaît. Ce n'est donc pas du hasard pur : c'est du **MAR**.

> **À retenir :** beaucoup de valeurs manquantes ne veut pas dire que les données sont mauvaises. Il faut chercher pourquoi elles manquent.

# 2. Que faire des valeurs manquantes ?

### Concept

Pour une case vide, on a trois choix :

| Choix | Ce qu'on fait | Exemple | Danger |
|---|---|---|---|
| **Supprimer** les lignes | on enlève toutes les lignes qui ont une case vide | on enlève les maisons sans surface de terrain | on perd des données, et les lignes qui restent peuvent être « particulières » |
| **Remplacer par une valeur calculée** | on remplit la case avec la moyenne ou la médiane de la colonne | on met la surface de terrain « typique » | on invente une valeur qui n'est peut-être pas vraie |
| **Remplacer par une valeur qui a du sens** | on remplit avec `0` ou « inconnu » | pas de terrain, donc `0` m² | il faut avoir compris pourquoi la case est vide |

> **Imputer** veut simplement dire « remplir une case vide avec une valeur choisie ».

### Démo : la surface de terrain des maisons

Les maisons ont peu de cases vides (3,5 %). On calcule la surface de terrain moyenne de 3 façons, puis on compare les résultats.

In [4]:
# Objectif : voir si la façon de traiter les cases vides change le résultat.
# On calcule la surface de terrain moyenne des maisons de 3 façons, puis on compare les 3 résultats.

# dvf["type_local"] == "Maison" garde seulement les lignes dont le type est Maison,
# puis ["surface_terrain"] garde uniquement cette colonne
maisons = dvf[dvf["type_local"] == "Maison"]["surface_terrain"]

# Combien de cases vides ? (isna() repère les cases vides, sum() les compte)
print("Manquants :", maisons.isna().sum(), "sur", len(maisons))

# Méthode 1 : dropna() supprime les maisons dont le terrain n'est pas renseigné, puis on fait la moyenne
print("Moyenne en supprimant les lignes :", round(maisons.dropna().mean(), 1))

# Méthode 2 : fillna(x) remplit les cases vides avec la valeur x.
# Ici x est la médiane (la valeur du milieu quand on trie toutes les surfaces)
print("Moyenne en remplaçant par la médiane :", round(maisons.fillna(maisons.median()).mean(), 1))

# Méthode 3 : on remplit les cases vides avec 0 (on suppose qu'il n'y a pas de terrain)
print("Moyenne en remplaçant par 0 :", round(maisons.fillna(0).mean(), 1))

Manquants : 17 sur 491
Moyenne en supprimant les lignes : 301.5
Moyenne en remplaçant par la médiane : 300.7
Moyenne en remplaçant par 0 : 291.0


**Lecture :** pour les maisons, il y a peu de cases vides (17 sur 491). Les trois méthodes donnent presque la même moyenne (entre 291 et 301 m²). Le choix n'a donc pas beaucoup d'importance ici.

### À vous : la même chose pour les appartements

Pour les appartements, il y a **beaucoup** de cases vides (88 %). On veut voir si les trois méthodes donnent encore le même résultat.

**À faire :** calculez la surface de terrain moyenne des appartements de trois façons :
1. en supprimant les lignes vides ;
2. en remplaçant les cases vides par la médiane ;
3. en remplaçant les cases vides par `0`.

**Questions :**
1. Les trois moyennes sont-elles proches ?
2. Quelle méthode choisissez-vous ? Pourquoi ?

In [5]:
# Objectif : refaire le même calcul que pour les maisons, mais avec les appartements,
# et voir si les 3 façons de traiter les manquants donnent encore le même résultat.

# Surface de terrain des appartements
apparts = dvf[dvf["type_local"] == "Appartement"]["surface_terrain"]

# Façon 1 : supprimer les lignes manquantes (dropna), puis faire la moyenne
moy_suppression = round(apparts.dropna().mean(), 1)

# Façon 2 : remplacer les manquants par la médiane (fillna), puis faire la moyenne
moy_mediane = round(apparts.fillna(apparts.median()).mean(), 1)

# Façon 3 : remplacer les manquants par 0 (fillna), puis faire la moyenne
moy_zero = round(apparts.fillna(0).mean(), 1)

print("Suppression :", moy_suppression)
print("Médiane     :", moy_mediane)
print("Zéro        :", moy_zero)

Suppression : 6796.0
Médiane     : 1121.0
Zéro        : 832.3


### Réponse

| Méthode | Moyenne |
|---|---:|
| Suppression des lignes | 6 796 m² |
| Remplacement par la médiane | 1 121 m² |
| Remplacement par `0` | 832 m² |

**Question 1 : les trois moyennes sont-elles proches ?**
Non. Elles sont très différentes, parce que 88 % des valeurs manquent.

**Question 2 : quelle méthode choisir ?**
Remplacer par `0`. Un appartement n'a pas de terrain à lui, donc `0` est la vraie valeur.

Pourquoi pas les deux autres :
- **Suppression :** il ne reste que 115 appartements sur 939, avec de grands terrains communs. Ils ne représentent pas tous les appartements.
- **Médiane :** on invente un terrain de 329 m² pour 824 appartements qui n'en ont pas.

> **À retenir :** on choisit la méthode selon la cause du manque, pas par habitude.

# 3. Repérer et diagnostiquer les anomalies

### Concept

Une **anomalie** (ou **valeur aberrante**) est une valeur très différente de toutes les autres. Exemple : un appartement vendu 1 € alors que les autres sont vendus plusieurs centaines de milliers d'euros.

Une anomalie n'est **pas toujours une erreur** : ce peut être une erreur de saisie, mais aussi un cas rare mais vrai. On suit donc trois étapes :
1. **repérer** la valeur bizarre ;
2. **comprendre** pourquoi elle est bizarre (on dit **diagnostiquer**) ;
3. seulement ensuite, décider quoi en faire (la supprimer, la corriger ou la garder).

Dans cette partie, on étudie le **prix au m²** des logements :

> prix au m² = prix de vente ÷ surface

Exemple : un appartement de 50 m² vendu 200 000 € coûte 200 000 ÷ 50 = 4 000 € le m².

### Première façon de repérer : la règle de l'IQR

L'IQR (*InterQuartile Range*, « écart interquartile ») est la règle qu'utilise le graphique **boxplot** (« boîte à moustaches ») pour marquer les points suspects.

**Étape 1.** On trie les valeurs de la plus petite à la plus grande, puis on les coupe en 4 groupes de même taille. Cela donne trois repères, appelés **quartiles** :
- `Q1` : la valeur sous laquelle se trouvent 25 % des observations ;
- `Q2` : la valeur du milieu (50 % des observations sont en dessous), c'est la **médiane** ;
- `Q3` : la valeur sous laquelle se trouvent 75 % des observations.

**Étape 2.** On calcule `IQR = Q3 - Q1`. C'est l'écart qui contient la moitié centrale des données.

**Étape 3.** Une valeur est suspecte si elle est **sous `Q1 - 1,5 × IQR`** ou **au-dessus de `Q3 + 1,5 × IQR`**. Le nombre 1,5 est une convention (une habitude des statisticiens), pas une loi.

**Exemple :** 9 notes triées : 4, 8, **9**, 10, **11**, 12, **13**, 14, 20.
- `Q1` = 9, `Q2` = 11, `Q3` = 13, donc `IQR` = 13 - 9 = 4 ;
- les bornes sont `9 - 1,5 × 4 = 3` et `13 + 1,5 × 4 = 19` ;
- la note 20 dépasse 19 : elle est suspecte.

### Deuxième façon de repérer : le bon sens

Un prix de 1 € pour une maison n'est pas un prix de marché, quelle que soit la statistique.

### Démo : l'IQR sur le prix au m²

In [6]:
# Objectif : trouver les logements dont le prix au m² est anormalement bas ou anormalement haut.
# Méthode : la règle de l'IQR (on calcule deux bornes, et tout prix en dehors est suspect).

# On garde les maisons et les appartements (isin = "est dans la liste")
# copy() fait une copie du tableau, pour pouvoir ajouter une colonne sans erreur
batis = dvf[dvf["type_local"].isin(["Maison", "Appartement"])].copy()

# Nouvelle colonne : le prix au m² = prix de vente divisé par la surface
batis["prix_m2"] = batis["valeur_fonciere"] / batis["surface_reelle_bati"]

# Q1 et Q3 : 25 % des prix sont sous Q1, 75 % des prix sont sous Q3
q1, q3 = batis["prix_m2"].quantile([0.25, 0.75])

# IQR : l'écart entre Q3 et Q1
iqr = q3 - q1

# Bornes : un prix en dehors de ces deux bornes est suspect
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Bornes : {borne_basse:.0f} à {borne_haute:.0f} euros/m2")

# On garde les lignes dont le prix est sous la borne basse OU (symbole |) au-dessus de la borne haute
suspects = batis[(batis["prix_m2"] < borne_basse) | (batis["prix_m2"] > borne_haute)]
print("Lignes suspectes :", len(suspects), "sur", len(batis))

Bornes : 226 à 9423 euros/m2
Lignes suspectes : 107 sur 1430


**Lecture :** l'IQR repère 107 lignes suspectes sur 1 430. Mais « suspect » ne veut pas dire « faux ». Il faut regarder les lignes une par une.

### À vous : ces prix sont-ils normaux ?

Parmi les prix au m², certains sont très petits ou très grands. On veut comprendre ce qui s'est passé.

Rappel : une même vente (`id_mutation`) peut avoir plusieurs lignes, une par lot.

**À faire :**
1. Affichez les 5 prix au m² les plus bas et les 5 les plus hauts.
2. Affichez toutes les lignes des transactions `2024-186755` et `2024-185518`.

**Questions :** pour chacune de ces deux transactions :
1. Qu'est-ce qui est bizarre ?
2. On la supprime, on la corrige ou on la garde ? Pourquoi ?

In [7]:
# Objectif : comprendre pourquoi certains prix au m² sont extrêmes,
# en regardant les lignes concernées.

# Colonnes à afficher
colonnes = ["id_mutation", "valeur_fonciere", "surface_reelle_bati", "type_local", "prix_m2"]

# Partie 1 : les prix au m² les plus bas et les plus hauts
# Étape 1 : trier du plus bas au plus haut prix au m² (sort_values)
tri = batis.sort_values("prix_m2")[colonnes]
# Étape 2 : afficher les 5 premières lignes (head) puis les 5 dernières (tail)
print(tri.head(5))
print(tri.tail(5))

# Partie 2 : les lignes des deux transactions (isin garde les lignes dont l'id est dans la liste)
batis[batis["id_mutation"].isin(["2024-186755", "2024-185518"])][colonnes]

      id_mutation  valeur_fonciere  surface_reelle_bati   type_local  \
3017  2024-186755              1.0                 54.0       Maison   
3018  2024-186755              1.0                 44.0       Maison   
3016  2024-186755              1.0                 38.0       Maison   
1202  2024-180888              1.0                 11.0  Appartement   
2285  2024-184380            460.0                 20.0  Appartement   

        prix_m2  
3017   0.018519  
3018   0.022727  
3016   0.026316  
1202   0.090909  
2285  23.000000  
      id_mutation  valeur_fonciere  surface_reelle_bati   type_local  \
838   2024-179588        2190476.0                 52.0  Appartement   
2646  2024-185518         800000.0                 18.0  Appartement   
1458  2024-181790         820000.0                 16.0       Maison   
837   2024-179588        2190476.0                 32.0  Appartement   
1705  2024-182588              NaN                 38.0       Maison   

           prix_m2  
838  

,id_mutation,valeur_fonciere,surface_reelle_bati,type_local,prix_m2
2640,2024-185518,800000.0,30.0,Appartement,26666.666667
2641,2024-185518,800000.0,22.0,Appartement,36363.636364
2642,2024-185518,800000.0,30.0,Appartement,26666.666667
2643,2024-185518,800000.0,31.0,Appartement,25806.451613
2644,2024-185518,800000.0,22.0,Appartement,36363.636364
2645,2024-185518,800000.0,22.0,Appartement,36363.636364
2646,2024-185518,800000.0,18.0,Appartement,44444.444444
3016,2024-186755,1.0,38.0,Maison,0.026316
3017,2024-186755,1.0,54.0,Maison,0.018519
3018,2024-186755,1.0,44.0,Maison,0.022727


### Réponse

**Transaction `2024-186755`**
- *Qu'est-ce qui est bizarre ?* Le prix est de **1 €** pour 3 maisons (0,02 € le m²).
- *Que fait-on ?* On la garde dans la base, mais on ne l'utilise pas pour étudier les prix. Ce n'est pas un vrai prix de vente : c'est une vente symbolique (par exemple dans une famille).

**Transaction `2024-185518`**
- *Qu'est-ce qui est bizarre ?* Le même prix de **800 000 €** est répété sur 7 lignes (7 petits appartements).
- *Que fait-on ?* On la **corrige**. 800 000 € est le prix de tout l'ensemble, pas de chaque appartement. En divisant par la surface d'un seul appartement, on trouve entre 26 000 et 44 000 € le m², ce qui est faux. Il faut additionner les surfaces des 7 lignes, puis diviser une seule fois.

> **À retenir :** deux prix extrêmes, deux causes différentes, deux traitements différents. On cherche la cause avant de supprimer.

### Concept : une deuxième méthode, le z-score

Le **z-score** mesure à quel point une valeur s'éloigne de la moyenne.
- L'**écart-type** est la « distance habituelle » entre les valeurs et la moyenne. Plus il est grand, plus les valeurs sont étalées.
- Le z-score d'une valeur est `z = (valeur - moyenne) / écart-type`. Il se lit : « à combien d'écarts-types de la moyenne se trouve cette valeur ».

**Exemple :** les notes 10, 12, 14, 16 et 18 ont pour moyenne 14 et pour écart-type environ 3,2. La note 18 a un z-score de (18 - 14) / 3,2 ≈ 1,3 : elle est à 1,3 écart-type de la moyenne, ce n'est pas bizarre.

**Règle usuelle :** une valeur est suspecte si `|z| > 3`. Le symbole `|z|` est la **valeur absolue** de z : on enlève le signe moins. Cela veut dire « z plus grand que 3, ou plus petit que -3 ».

### À vous : deux méthodes pour trouver les avis très longs

On veut repérer les avis dont la longueur est très différente des autres. On a deux méthodes :
- **le z-score** : l'avis est suspect si `|z| > 3` ;
- **l'IQR** : l'avis est suspect s'il est plus long que `Q3 + 1,5 × IQR`.

**À faire :** calculez la longueur de chaque avis (`TextLength`, en nombre de caractères), puis comptez les avis suspects avec chaque méthode.

**Questions :**
1. Les deux méthodes trouvent-elles le même nombre d'avis ?
2. Pourquoi ?
3. Ces avis très longs sont-ils des erreurs ?

In [8]:
# Objectif : repérer les avis anormalement longs avec deux méthodes (z-score et IQR),
# puis comparer le nombre d'avis trouvé par chacune.

# Chargement des avis Amazon
amazon = pd.read_csv(f"{DATA}/amazon_reviews_sample.csv")

# Longueur de chaque avis, en nombre de caractères
amazon["TextLength"] = amazon["Text"].str.len()
L = amazon["TextLength"]

# Méthode 1 : le z-score
# Étape 1 : calculer z pour chaque avis
z = (L - L.mean()) / L.std()
# Étape 2 : compter les avis où la valeur absolue de z est supérieure à 3
nb_z = int((z.abs() > 3).sum())

# Méthode 2 : l'IQR
# Étape 1 : calculer Q1, Q3 et l'IQR
q1, q3 = L.quantile([0.25, 0.75])
iqr = q3 - q1
# Étape 2 : compter les avis plus longs que Q3 + 1.5 * IQR
nb_iqr = int((L > q3 + 1.5 * iqr).sum())

print("z-score :", nb_z)
print("IQR     :", nb_iqr)

z-score : 184
IQR     : 713


### Réponse

- **z-score :** 184 avis (seuil à environ 1 730 caractères)
- **IQR :** 713 avis (seuil à environ 1 030 caractères)

**Question 1 : les deux méthodes trouvent-elles le même nombre d'avis ?**
Non. L'IQR en trouve presque 4 fois plus.

**Question 2 : pourquoi ?**
Beaucoup d'avis sont courts et quelques-uns sont très longs (jusqu'à plus de 8 000 caractères). Ces avis très longs font monter l'écart-type, donc le seuil du z-score est plus haut. L'IQR n'est pas gêné par ces avis très longs : son seuil est plus bas.

**Question 3 : ces avis très longs sont-ils des erreurs ?**
Non. Ce sont des clients qui écrivent beaucoup.

> **À retenir :** le z-score marche bien quand les données sont à peu près symétriques. Quand quelques valeurs sont très grandes, préférez l'IQR.

# 4. Isolation Forest

### Concept

L'IQR et le z-score regardent **une seule colonne à la fois**. Mais un logement peut avoir un prix normal et une surface normale, et pourtant une **combinaison** bizarre : par exemple une maison de 20 m² vendue 2 millions d'euros. Chaque valeur seule paraît normale, mais les deux ensemble sont étranges.

L'**Isolation Forest** (« forêt d'isolement ») est un **modèle** (un programme qui apprend à partir des données) qui repère ces cas en regardant plusieurs colonnes en même temps.

Comment ça marche, en simple :
1. Il pose au hasard des questions comme « le prix est-il supérieur à 300 000 € ? » ou « la surface est-elle inférieure à 80 m² ? ». Chaque question coupe les données en deux. Une suite de questions s'appelle un **arbre**.
2. Il refait cela dans beaucoup d'arbres : c'est une **forêt**.
3. Un logement **ordinaire** est au milieu des autres : il faut beaucoup de questions pour l'isoler.
4. Un logement **bizarre** est seul dans son coin : il est isolé en très peu de questions.

Moins il faut de questions pour isoler un logement, plus il est suspect.

Référence : F. T. Liu, K. M. Ting, Z.-H. Zhou, [*Isolation Forest*](https://www.semanticscholar.org/paper/Isolation-Forest-Liu-Ting/00a1077d298f2917d764eb729ab1bc86af3bd241), IEEE ICDM, 2008.

### Démo

On regarde le prix et la surface **en même temps**.

In [9]:
# Objectif : repérer les logements bizarres en regardant le prix ET la surface en même temps
# (l'IQR ne regarde qu'une seule colonne à la fois).

# On importe le modèle Isolation Forest
from sklearn.ensemble import IsolationForest

# On utilise deux colonnes à la fois : le prix et la surface
# (dropna enlève les lignes qui ont une case vide)
X = batis[["valeur_fonciere", "surface_reelle_bati"]].dropna()

# contamination : la part de logements bizarres qu'on s'attend à trouver (0.02 = 2 sur 100)
# random_state : pour avoir le même résultat à chaque lancement
iso = IsolationForest(contamination=0.02, random_state=42)

# fit_predict : le modèle observe les données (fit), puis donne une étiquette à chaque ligne (predict)
# -1 = logement bizarre, 1 = logement normal
etiquettes = iso.fit_predict(X)

# On compte les logements bizarres
print("Anomalies détectées :", (etiquettes == -1).sum(), "sur", len(X))

# On affiche ces logements, triés par prix
X[etiquettes == -1].sort_values("valeur_fonciere").head(8)

Anomalies détectées : 29 sur 1429


,valeur_fonciere,surface_reelle_bati
114,19200.0,148.0
1903,316000.0,199.0
2014,650000.0,221.0
3658,730000.0,190.0
3657,730000.0,210.0
690,800000.0,209.0
435,894200.0,200.0
2565,970000.0,235.0


**Lecture du résultat**

Le modèle a trouvé **29 anomalies** sur 1 429 logements, soit 2 % (c'est ce qu'on a demandé avec `contamination=0.02`).

Le tableau montre les 8 anomalies **les moins chères**. Chaque ligne est un logement, avec son prix (`valeur_fonciere`) et sa surface (`surface_reelle_bati`). Pour savoir si un logement est bizarre, on le compare à un logement « typique » : le prix médian est de 255 000 € et la surface médiane de 61 m².

Ce qu'on voit :
- **7 de ces 8 logements sont très grands** : entre 190 et 235 m² (contre 61 m² en médiane), pour des prix de 316 000 à 970 000 €. Le modèle les a repérés parce que la combinaison « très grande surface et prix élevé » est rare. Ce sont des cas rares mais plausibles : de grandes maisons.
- **Une ligne est plus étrange** : 19 200 € pour 148 m², soit 130 € le m², alors que le prix médian est d'environ 4 800 € le m². C'est presque gratuit : il faudrait vérifier cette vente.

Dans la liste complète (qu'on n'affiche pas ici), on trouve aussi des petites surfaces à des prix énormes, par exemple 32 m² à 2 190 476 €. C'est la transaction `2024-179588` : le prix de toute la vente est recopié sur chaque lot, comme dans l'exemple de la partie 3.

> **À retenir :** l'Isolation Forest ne dit pas « erreur ». Il dit « ce logement est isolé, regardez-le ». Parmi les 29 anomalies, il y a des cas rares mais vrais (grandes maisons), un prix suspect à vérifier et des prix recopiés. Comme en partie 3 : on **repère**, puis on **diagnostique**.

### À vous : le paramètre `contamination`

Dans la démo, on a donné `contamination=0.02`. Cela veut dire au modèle : « parmi les logements, environ 2 sur 100 sont des anomalies ». Il a donc gardé les 2 % de logements les plus isolés (29 sur 1 429).

Imaginez que le modèle donne à chaque logement une **note de bizarrerie**. Le paramètre `contamination` décide à partir de quelle note on dit « c'est une anomalie ». On veut voir ce qui se passe si on change cette limite.

**À faire :** relancez l'Isolation Forest deux fois :
1. avec `contamination=0.05` (5 sur 100) ;
2. avec `contamination=0.005` (5 sur 1 000).

Comptez les anomalies à chaque fois.

**Questions :**
1. Combien d'anomalies trouve-t-on avec `0.05` ? Et avec `0.005` ?
2. Quand on augmente `contamination`, le modèle trouve-t-il plus d'anomalies ou moins ?
3. Ce paramètre change-t-il la façon de repérer les logements bizarres, ou seulement le nombre de logements qu'on garde ?

In [10]:
# Objectif : voir si le paramètre contamination change le nombre d'anomalies trouvées.

# Pour chaque valeur de contamination :
for c in [0.05, 0.005]:
    # Étape 1 : créer le modèle
    iso = IsolationForest(contamination=c, random_state=42)
    # Étape 2 : l'appliquer aux données X (-1 = anomalie, 1 = normal)
    etiquettes = iso.fit_predict(X)
    # Étape 3 : compter les anomalies
    print("contamination =", c, "->", (etiquettes == -1).sum(), "anomalies")

contamination = 0.05 -> 70 anomalies


contamination = 0.005 -> 8 anomalies


### Réponse

**Question 1 : combien d'anomalies ?**
- `contamination=0.05` : **70** anomalies
- `contamination=0.005` : **8** anomalies
- (pour rappel, `0.02` : 29 anomalies)

**Question 2 : plus ou moins d'anomalies quand on augmente `contamination` ?**
Plus. Plus la valeur est grande, plus le modèle en trouve : 8, puis 29, puis 70.

**Question 3 : qu'est-ce que ce paramètre change ?**
Seulement le **nombre** de logements qu'on garde. Le modèle donne la même note de bizarrerie à chaque logement, quelle que soit la valeur de `contamination`. Ce paramètre ne fait que placer la limite : on garde les `x %` de logements qui ont la note la plus élevée.

Ensuite, c'est à vous de regarder les lignes retenues pour savoir si ce sont de vraies erreurs (comme dans la partie 3).

# 5. Bilan

| Situation | Question à se poser | Ce qu'on peut faire |
|---|---|---|
| Une case est vide | Pourquoi est-elle vide ? (hasard, lien avec une autre colonne, ou la valeur n'existe pas) | supprimer la ligne, remplir avec la médiane, ou remplir avec `0` / « inconnu » |
| Une valeur est très grande ou très petite | Est-ce une erreur, un cas rare mais vrai, ou un effet de la façon dont les données sont rangées ? | la corriger, l'écarter de l'analyse, ou la garder |
| Une combinaison de plusieurs colonnes paraît étrange | Chaque colonne seule est-elle normale ? | Isolation Forest, puis diagnostic des lignes trouvées |

> **Prochaine séance :** réduction de dimension (PCA, t-SNE, UMAP).